<a href="https://colab.research.google.com/github/busycaesar/Embeddings_And_Cosine_Similarity/blob/Master/TorontoJS/main.colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install required dependencies

In [11]:
%pip install pypdf langchain-community langchain-google-community

Import environment variables

In [14]:
from google.colab import userdata

GEMINI_API_KEYS = userdata.get('GEMINI_API_KEYS')
PROJECT_ID = userdata.get('PROJECT_ID')
DATASET = userdata.get('DATASET')
TABLE = userdata.get('TABLE')
REGION = userdata.get('REGION')

from google.colab import auth
auth.authenticate_user()

MessageError: Error: credential propagation was unsuccessful

Fetch the data

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://services.google.com/fh/files/misc/startup_technical_guide_ai_agents_final.pdf")

documents = loader.load()

Split the data into chunks

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=20)

chunks = text_splitter.split_documents(documents)

Store the data into vector database

In [12]:
from langchain_google_community import BigQueryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

bq_vector_store = BigQueryVectorStore(
    project_id=PROJECT_ID,
    dataset_name=DATASET,
    table_name=TABLE,
    location=REGION,
    embedding=embedding_model
)

bq_vector_store.add_documents(chunks)

RefreshError: ("Failed to retrieve http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/?recursive=true from the Google Compute Engine metadata service. Status: 404 Response:\nb''", <google.auth.transport.requests._Response object at 0x7f6df8f6e030>)

User's query

In [ ]:
user_query = "What are the Core components for building AI agents?"

Fetch the relevant chunk of data

In [ ]:
retrieved_docs = bq_vector_store.as_retriever().invoke(user_query)

retrieved_docs = " ".join([doc.page_content for doc in retrieved_docs])

Create prompt template

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["prompt", "relevant_chunk_of_data"],
    template=
    """
        Use the following pieces of context to answer the question at the end.

        Context: {relevant_chunk_of_data}

        User's Question: {prompt}
    """
)

Chain the prompt template with LLM and invoke it to generate the response

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from IPython.display import clear_output

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", google_api_key=GEMINI_API_KEYS)

chain = prompt_template | llm

response = chain.invoke({
    "prompt": user_query,
    "relevant_chunk_of_data": retrieved_docs
})

clear_output(wait=True)

print(response.content)